# Temporary Legacy Colab Export Migration

This notebook migrates the old flat Colab export layout under `pcc_colab_outputs/` into the new nested structure:

- `pcc_colab_outputs/run_YYYYMMDD_HHMMSS/run_1/train/`
- `pcc_colab_outputs/run_YYYYMMDD_HHMMSS/run_1/discovery/<decode_type>__YYYYMMDD_HHMMSS/`

It copies by default so the legacy `train_run_*` and `discover_run_*` folders remain untouched until you verify the migrated result.

In [ ]:
# Configuration - edit these values, then Run All

LEGACY_EXPORT_ROOT = '/content/drive/MyDrive/pcc_colab_outputs'

LEGACY_TRAIN_EXPORT = 'train_run_20260407_005201'  # required legacy training folder name
LEGACY_DISCOVERY_EXPORT = 'discover_run_20260407_193057'  # optional legacy discovery folder name; set to None to migrate training only

NEW_RUN_ID = None  # None = derive from LEGACY_TRAIN_EXPORT, e.g. run_20260407_005201
NEW_RUN_NAME = 'run_1'  # keep the new nested layout consistent with the current notebooks

DISCOVERY_ATTEMPT_TIMESTAMP = None  # None = derive from LEGACY_DISCOVERY_EXPORT, e.g. 20260407_193057
DISCOVERY_DECODE_TYPE = None  # None = infer from the legacy discovery artifact; set manually only if inference fails

OVERWRITE_EXISTING_TARGETS = False  # True = replace an already-migrated target folder
REMOVE_LEGACY_EXPORTS_AFTER_COPY = False  # keep False until you verify the new layout and want to clean up the old flat folders

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import re
import shutil
import yaml

drive.mount('/content/drive')

export_root = Path(LEGACY_EXPORT_ROOT)
print('Legacy export root:', export_root)

In [ ]:
def _extract_timestamp(name: str, prefix: str) -> str | None:
    if not name.startswith(prefix):
        return None
    suffix = name[len(prefix):].strip()
    return suffix or None


def _sanitize_segment(value: str) -> str:
    sanitized = re.sub(r'[^A-Za-z0-9._-]+', '-', str(value).strip())
    sanitized = sanitized.strip('.-_')
    return sanitized or 'discovery'


def _derive_run_id(legacy_train_name: str, explicit_run_id: str | None) -> str:
    if explicit_run_id is not None:
        explicit = str(explicit_run_id).strip()
        if not explicit:
            raise ValueError('NEW_RUN_ID must not be empty when provided.')
        return explicit
    suffix = _extract_timestamp(legacy_train_name, 'train_run_')
    if suffix is None:
        raise ValueError(
            f"Could not derive NEW_RUN_ID from '{legacy_train_name}'. Set NEW_RUN_ID manually."
        )
    return f'run_{suffix}'


def _derive_attempt_timestamp(legacy_discovery_name: str | None, explicit_timestamp: str | None) -> str | None:
    if explicit_timestamp is not None:
        explicit = str(explicit_timestamp).strip()
        if not explicit:
            raise ValueError('DISCOVERY_ATTEMPT_TIMESTAMP must not be empty when provided.')
        return explicit
    if legacy_discovery_name is None:
        return None
    suffix = _extract_timestamp(legacy_discovery_name, 'discover_run_')
    if suffix is None:
        raise ValueError(
            f"Could not derive DISCOVERY_ATTEMPT_TIMESTAMP from '{legacy_discovery_name}'. Set it manually."
        )
    return suffix


def _discovery_artifact_candidates(checkpoint_dir: Path) -> list[Path]:
    if not checkpoint_dir.exists():
        return []
    candidates = []
    for path in checkpoint_dir.glob('*_discovery.json'):
        if path.name.endswith('_decode_coverage.json'):
            continue
        if path.name.endswith('_cluster_summary.json'):
            continue
        if path.name.endswith('_validation.json'):
            continue
        candidates.append(path)
    return sorted(candidates)


def _find_legacy_discovery_artifact(legacy_discovery_dir: Path) -> Path:
    checkpoint_dir = legacy_discovery_dir / 'checkpoints'
    candidates = _discovery_artifact_candidates(checkpoint_dir)
    if not candidates:
        raise FileNotFoundError(
            f'Could not find a legacy discovery artifact under {checkpoint_dir}. '
            'Expected a *_discovery.json file in checkpoints/.'
        )
    return candidates[-1]


def _infer_decode_type(discovery_artifact_path: Path, explicit_decode_type: str | None) -> str:
    if explicit_decode_type is not None:
        explicit = str(explicit_decode_type).strip()
        if not explicit:
            raise ValueError('DISCOVERY_DECODE_TYPE must not be empty when provided.')
        return explicit
    payload = json.loads(discovery_artifact_path.read_text(encoding='utf-8'))
    for container in (
        payload.get('decoder_summary') or {},
        ((payload.get('config_snapshot') or {}).get('discovery') or {}),
    ):
        target_label = container.get('target_label')
        if isinstance(target_label, str) and target_label.strip():
            return target_label
    raise ValueError(
        'Could not infer the discovery decode type from the legacy artifact. '
        'Set DISCOVERY_DECODE_TYPE manually and rerun the notebook.'
    )


def _infer_checkpoint_name(legacy_train_dir: Path, discovery_artifact_path: Path | None) -> str | None:
    checkpoint_dir = legacy_train_dir / 'checkpoints'
    checkpoint_candidates = sorted(checkpoint_dir.glob('*.pt'))
    if checkpoint_candidates:
        best_candidates = [path for path in checkpoint_candidates if path.name.endswith('_best.pt')]
        return (best_candidates[-1] if best_candidates else checkpoint_candidates[-1]).name
    if discovery_artifact_path is None:
        return None
    stem = discovery_artifact_path.stem
    marker = '_discovery'
    if marker in stem:
        return stem.split(marker, 1)[0] + '.pt'
    return None


def _collect_legacy_discovery_files(legacy_discovery_dir: Path) -> list[Path]:
    patterns = (
        '*_discovery.json',
        '*_decode_coverage.json',
        '*_cluster_summary.json',
        '*_cluster_summary.csv',
        '*_validation.json',
        '*_validation.csv',
        '*_discover_run_manifest.json',
        '*_validate_run_manifest.json',
    )
    discovered = {}
    for pattern in patterns:
        for path in legacy_discovery_dir.rglob(pattern):
            discovered[str(path.resolve())] = path
    return [discovered[key] for key in sorted(discovered)]


def _write_reconstructed_discovery_runtime(discovery_artifact_path: Path, target_dir: Path) -> Path | None:
    payload = json.loads(discovery_artifact_path.read_text(encoding='utf-8'))
    config_snapshot = payload.get('config_snapshot')
    if not isinstance(config_snapshot, dict):
        return None
    runtime_path = target_dir / 'colab_discovery_runtime_experiment.yaml'
    runtime_path.write_text(yaml.safe_dump(config_snapshot, sort_keys=False), encoding='utf-8')
    return runtime_path


def _prepare_target_dir(target_dir: Path) -> None:
    if target_dir.exists():
        if not OVERWRITE_EXISTING_TARGETS:
            raise FileExistsError(
                f'Target already exists: {target_dir}. Set OVERWRITE_EXISTING_TARGETS = True to replace it.'
            )
        shutil.rmtree(target_dir)
    target_dir.parent.mkdir(parents=True, exist_ok=True)


def _print_tree(root: Path, max_depth: int = 3) -> None:
    root = root.resolve()
    print(root)
    for path in sorted(root.rglob('*')):
        relative = path.relative_to(root)
        if len(relative.parts) > max_depth:
            continue
        indent = '  ' * (len(relative.parts) - 1)
        suffix = '/' if path.is_dir() else ''
        print(f'{indent}- {relative.name}{suffix}')

In [ ]:
legacy_train_dir = export_root / LEGACY_TRAIN_EXPORT
legacy_discovery_dir = export_root / LEGACY_DISCOVERY_EXPORT if LEGACY_DISCOVERY_EXPORT else None

if not legacy_train_dir.is_dir():
    raise FileNotFoundError(f'Missing legacy training export: {legacy_train_dir}')
if legacy_discovery_dir is not None and not legacy_discovery_dir.is_dir():
    raise FileNotFoundError(f'Missing legacy discovery export: {legacy_discovery_dir}')

new_run_id = _derive_run_id(legacy_train_dir.name, NEW_RUN_ID)
target_train_dir = export_root / new_run_id / NEW_RUN_NAME / 'train'

legacy_discovery_artifact = None
discovery_decode_type = None
discovery_attempt_timestamp = None
target_discovery_dir = None

if legacy_discovery_dir is not None:
    legacy_discovery_artifact = _find_legacy_discovery_artifact(legacy_discovery_dir)
    discovery_decode_type = _infer_decode_type(legacy_discovery_artifact, DISCOVERY_DECODE_TYPE)
    discovery_attempt_timestamp = _derive_attempt_timestamp(
        legacy_discovery_dir.name,
        DISCOVERY_ATTEMPT_TIMESTAMP,
    )
    target_discovery_dir = (
        export_root
        / new_run_id
        / NEW_RUN_NAME
        / 'discovery'
        / f"{_sanitize_segment(discovery_decode_type)}__{discovery_attempt_timestamp}"
    )

if target_train_dir.exists() and not OVERWRITE_EXISTING_TARGETS:
    raise FileExistsError(
        f'Target training directory already exists: {target_train_dir}. '
        'Set OVERWRITE_EXISTING_TARGETS = True to replace it.'
    )
if target_discovery_dir is not None and target_discovery_dir.exists() and not OVERWRITE_EXISTING_TARGETS:
    raise FileExistsError(
        f'Target discovery directory already exists: {target_discovery_dir}. '
        'Set OVERWRITE_EXISTING_TARGETS = True to replace it.'
    )

checkpoint_name = _infer_checkpoint_name(legacy_train_dir, legacy_discovery_artifact)

migration_plan = {
    'legacy_export_root': str(export_root),
    'legacy_train_export': legacy_train_dir.name,
    'legacy_discovery_export': legacy_discovery_dir.name if legacy_discovery_dir is not None else None,
    'new_run_id': new_run_id,
    'new_run_name': NEW_RUN_NAME,
    'target_train_dir': str(target_train_dir),
    'target_discovery_dir': str(target_discovery_dir) if target_discovery_dir is not None else None,
    'discovery_decode_type': discovery_decode_type,
    'discovery_attempt_timestamp': discovery_attempt_timestamp,
    'checkpoint_name': checkpoint_name,
    'overwrite_existing_targets': OVERWRITE_EXISTING_TARGETS,
    'remove_legacy_exports_after_copy': REMOVE_LEGACY_EXPORTS_AFTER_COPY,
}

print(json.dumps(migration_plan, indent=2))
print('\nLegacy train source contents:')
_print_tree(legacy_train_dir, max_depth=2)
if legacy_discovery_dir is not None:
    print('\nLegacy discovery source contents:')
    _print_tree(legacy_discovery_dir, max_depth=2)

print('\nPreflight complete. Run the next cell to copy into the new nested layout.')

In [ ]:
_prepare_target_dir(target_train_dir)
shutil.copytree(legacy_train_dir, target_train_dir)
print('Copied training bundle to:', target_train_dir)

discovery_runtime_path = None
discovery_metadata_path = None

if legacy_discovery_dir is not None:
    _prepare_target_dir(target_discovery_dir)
    target_discovery_dir.mkdir(parents=True, exist_ok=True)

    copied_count = 0
    for src in _collect_legacy_discovery_files(legacy_discovery_dir):
        relative = src.relative_to(legacy_discovery_dir)
        if src.suffix == '.pt':
            continue
        dst = target_discovery_dir / relative
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied_count += 1

    discovery_runtime_path = _write_reconstructed_discovery_runtime(
        legacy_discovery_artifact,
        target_discovery_dir,
    )

    metadata = {
        'training_run_id': new_run_id,
        'run_name': NEW_RUN_NAME,
        'decode_type': discovery_decode_type,
        'checkpoint_name': checkpoint_name,
        'attempt_timestamp': discovery_attempt_timestamp,
        'migrated_from_legacy_layout': True,
        'legacy_train_export': legacy_train_dir.name,
        'legacy_discovery_export': legacy_discovery_dir.name,
        'source_discovery_artifact_path': str(legacy_discovery_artifact),
        'exported_discovery_runtime_config_path': str(discovery_runtime_path) if discovery_runtime_path is not None else None,
    }
    discovery_metadata_path = target_discovery_dir / 'discovery_export_metadata.json'
    discovery_metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

    print(f'Copied {copied_count} discovery and validation files to: {target_discovery_dir}')
    if discovery_runtime_path is not None:
        print('Reconstructed discovery runtime config:', discovery_runtime_path)
    else:
        print('No config_snapshot found in the legacy discovery artifact, so no discovery runtime YAML was reconstructed.')
    print('Wrote discovery metadata:', discovery_metadata_path)

if REMOVE_LEGACY_EXPORTS_AFTER_COPY:
    shutil.rmtree(legacy_train_dir)
    print('Removed legacy training export:', legacy_train_dir)
    if legacy_discovery_dir is not None:
        shutil.rmtree(legacy_discovery_dir)
        print('Removed legacy discovery export:', legacy_discovery_dir)

print('\nMigration finished.')

In [ ]:
new_run_root = export_root / new_run_id / NEW_RUN_NAME
print('Migrated run root:')
_print_tree(new_run_root, max_depth=4)

print('\nTop-level pcc_colab_outputs contents:')
for path in sorted(export_root.iterdir()):
    suffix = '/' if path.is_dir() else ''
    print(f'- {path.name}{suffix}')